
# Hybridized Evolutionary Factoring (HEF)

This notebook recreates the original prime factorization app as an interactive Jupyter experience.

- Enter a positive integer greater than 1
- Use preset values or custom input
- Compute the prime factorization and execution time
- Review recent calculation history


In [1]:

import math
import time
from collections import Counter

from IPython.display import HTML, Markdown, display
import ipywidgets as widgets

PRESETS = [
    ("12", "12", "2² × 3"),
    ("360", "360", "2³ × 3² × 5"),
    ("2,310", "2310", "2 × 3 × 5 × 7 × 11"),
    ("104,729", "104729", "Prime"),
    ("1,234,567", "1234567", "127 × 9721"),
    ("982,451,653", "982451653", "Large prime"),
    ("123,456,789,012", "123456789012", "2² × 3 × 13 × 23 × 3433367"),
]

history = []


In [2]:

def gcd(a, b):
    while b:
        a, b = b, a % b
    return abs(a)


def mod_pow(base, exp, mod):
    result = 1
    base %= mod
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % mod
        base = (base * base) % mod
        exp //= 2
    return result


def is_prime_bigint(n):
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False

    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1

    for a in [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]:
        if n <= a:
            break
        x = mod_pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(1, s):
            x = (x * x) % n
            if x == n - 1:
                break
        else:
            return False
    return True


def pollards_rho(n, iteration_counter):
    if n % 2 == 0:
        return 2
    if is_prime_bigint(n):
        return n

    c = 1
    while True:
        x = 2
        y = 2
        d = 1

        def f(v):
            return (pow(v, 2, n) + c) % n

        while d == 1:
            iteration_counter['count'] += 1
            x = f(x)
            y = f(f(y))
            d = math.gcd(abs(x - y), n)

        if d != n:
            return d
        c += 1
        if c > 100:
            for i in range(3, int(math.isqrt(n)) + 1, 2):
                iteration_counter['count'] += 1
                if n % i == 0:
                    return i
            return n


def factorize_recursive(n, factors, iteration_counter):
    if n <= 1:
        return
    if is_prime_bigint(n):
        factors.append(n)
        return

    small_primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
    for p in small_primes:
        if n % p == 0:
            while n % p == 0:
                factors.append(p)
                n //= p
                iteration_counter['count'] += 1
            if n == 1:
                return
            if is_prime_bigint(n):
                factors.append(n)
                return

    divisor = pollards_rho(n, iteration_counter)
    factorize_recursive(divisor, factors, iteration_counter)
    factorize_recursive(n // divisor, factors, iteration_counter)


In [3]:

def run_hef_algorithm(raw_value):
    cleaned = str(raw_value).replace(',', '').strip()
    if not cleaned:
        return {
            'success': False,
            'error': 'Please enter a number.',
            'inputNumber': cleaned,
            'canonicalForm': '',
            'productForm': '',
            'factorPowers': [],
            'factors': [],
            'isPrime': False,
            'isSemiprime': False,
            'totalPrimeFactors': 0,
            'distinctPrimeFactors': 0,
            'divisorsCount': 0,
            'iterations': 0,
            'execution_time': 0,
        }

    try:
        n = int(cleaned)
    except ValueError:
        return {
            'success': False,
            'error': 'Please enter a valid integer.',
            'inputNumber': cleaned,
            'canonicalForm': '',
            'productForm': '',
            'factorPowers': [],
            'factors': [],
            'isPrime': False,
            'isSemiprime': False,
            'totalPrimeFactors': 0,
            'distinctPrimeFactors': 0,
            'divisorsCount': 0,
            'iterations': 0,
            'execution_time': 0,
        }

    if n <= 1:
        return {
            'success': False,
            'error': 'Integer must be greater than 1.',
            'inputNumber': cleaned,
            'canonicalForm': '',
            'productForm': '',
            'factorPowers': [],
            'factors': [],
            'isPrime': False,
            'isSemiprime': False,
            'totalPrimeFactors': 0,
            'distinctPrimeFactors': 0,
            'divisorsCount': 0,
            'iterations': 0,
            'execution_time': 0,
        }

    start = time.perf_counter()
    iteration_counter = {'count': 0}
    factors = []
    factorize_recursive(n, factors, iteration_counter)
    elapsed_ms = max(0.001, (time.perf_counter() - start) * 1000)
    factors.sort()

    counts = Counter(factors)
    factor_powers = [
        {'prime': prime, 'power': power}
        for prime, power in sorted(counts.items())
    ]
    canonical = ' × '.join(
        f'{prime}^{power}' if power > 1 else str(prime)
        for prime, power in [(entry['prime'], entry['power']) for entry in factor_powers]
    )
    product = ' × '.join(str(f) for f in factors)
    divisors_count = math.prod(power + 1 for _, power in counts.items())
    is_prime = len(factors) == 1
    is_semiprime = len(factors) == 2

    return {
        'success': True,
        'error': None,
        'inputNumber': cleaned,
        'canonicalForm': canonical,
        'productForm': product,
        'factorPowers': factor_powers,
        'factors': factors,
        'isPrime': is_prime,
        'isSemiprime': is_semiprime,
        'totalPrimeFactors': len(factors),
        'distinctPrimeFactors': len(counts),
        'divisorsCount': divisors_count,
        'iterations': max(1, iteration_counter['count']),
        'execution_time': round(elapsed_ms, 4),
    }


In [4]:

def render_result(result):
    if not result['success']:
        return HTML(f"""
            <div style='padding: 16px; border: 1px solid #ef4444; border-radius: 12px; background: rgba(239,68,68,0.08); color: #fecaca'>
                <strong>Error:</strong> {result['error']}
            </div>
        """)

    cards_html = ''.join(
        f"<div style='padding: 12px 14px; border-radius: 10px; background: rgba(15,23,42,0.8); border: 1px solid #334155; margin: 6px; min-width: 130px;'><div style='font-size: 0.7rem; text-transform: uppercase; color: #94a3b8; letter-spacing: 0.08em'>{label}</div><div style='font-size: 1.25rem; font-weight: 700; color: white; margin-top: 6px'>{value}</div></div>"
        for label, value in [
            ('Total Prime Factors', result['totalPrimeFactors']),
            ('Distinct Primes', result['distinctPrimeFactors']),
            ('Divisors', result['divisorsCount']),
            ('Iterations', f"{result['iterations']:,}"),
            ('Execution Time', f"{result['execution_time']:.4f} ms"),
            ('Classification', 'Prime' if result['isPrime'] else 'Semiprime' if result['isSemiprime'] else 'Composite'),
        ]
    )

    factor_badges = ''.join(
        f"<span style='padding: 6px 10px; border-radius: 999px; background: #0f172a; border: 1px solid #334155; color: #67e8f9; font-family: monospace; margin: 4px'>{term['prime']}{f'^{term['power']}' if term['power'] > 1 else ''}</span>"
        for term in result['factorPowers']
    )

    return HTML(f"""
        <div style='padding: 18px; border: 1px solid #334155; border-radius: 12px; background: rgba(15,23,42,0.7); color: white;'>
            <h3 style='margin: 0 0 10px; color: #e2e8f0'>Canonical prime factorization</h3>
            <div style='font-size: 1.5rem; font-weight: 700; color: #f8fafc; margin-bottom: 8px'>{result['inputNumber']} = {result['canonicalForm']}</div>
            <div style='font-size: 0.9rem; color: #94a3b8; margin-bottom: 14px'>Expanded product: {result['inputNumber']} = {result['productForm']}</div>
            <div style='display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 18px'>{factor_badges}</div>
            <div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 8px'>{cards_html}</div>
        </div>
    """)


In [5]:

def render_history():
    if not history:
        history_box.clear_output(wait=True)
        with history_box:
            display(Markdown('No factorization records yet.'))
        return

    history_box.clear_output(wait=True)
    with history_box:
        display(HTML(
            '<table style="width:100%; border-collapse: collapse; color:#e2e8f0; font-family: sans-serif;">'
            + ''.join(
                f"<tr><td style='padding: 8px; border-bottom: 1px solid #334155'>{row['inputNumber']}</td>"
                f"<td style='padding: 8px; border-bottom: 1px solid #334155'>{row['canonicalForm']}</td>"
                f"<td style='padding: 8px; border-bottom: 1px solid #334155'>{row['execution_time']:.4f} ms</td>"
                f"<td style='padding: 8px; border-bottom: 1px solid #334155'>{row['status']}</td></tr>"
                for row in history[-10:]
            )
            + '</table>'
        ))


def on_calculate(_):
    result = run_hef_algorithm(input_text.value)
    result['status'] = 'Success' if result['success'] else 'Failure'
    history.append({
        'inputNumber': result['inputNumber'],
        'canonicalForm': result['canonicalForm'],
        'execution_time': result['execution_time'],
        'status': result['status'],
    })

    result_box.clear_output(wait=True)
    with result_box:
        display(render_result(result))
    render_history()


def on_preset_change(change):
    if change['new']:
        input_text.value = change['new']
        preset_dropdown.value = ''


input_text = widgets.Text(
    value='',
    placeholder='Example: 360, 2310, 123456789012',
    description='Integer',
    layout=widgets.Layout(width='100%')
)

preset_dropdown = widgets.Dropdown(
    options=[('Select preset', '')] + [(label, value) for label, value, _ in PRESETS],
    value='',
    description='Preset',
    layout=widgets.Layout(width='260px')
)

calculate_button = widgets.Button(description='Calculate prime factors')
calculate_button.on_click(on_calculate)
preset_dropdown.observe(on_preset_change, names='value')

result_box = widgets.Output()
history_box = widgets.Output()

ui = widgets.VBox([
    widgets.HTML('<h2 style="margin:0 0 10px">Prime Factorization & Benchmarking</h2>'),
    widgets.HBox([input_text]),
    widgets.HBox([preset_dropdown, calculate_button]),
    widgets.HTML('<div style="color:#94a3b8; font-size: 12px; margin-bottom: 8px">Tip: commas are accepted and ignored.</div>'),
    result_box,
    widgets.HTML('<h3 style="margin: 18px 0 8px">Recent History</h3>'),
    history_box,
])

with result_box:
    display(HTML("<div style='padding: 18px; border: 1px solid #334155; border-radius: 12px; background: rgba(15,23,42,0.7); color: #94a3b8'>No factorization has been run yet.</div>"))

with history_box:
    display(Markdown('No results yet.'))

display(ui)
